# 02 — Train on Kaggle

Launcher only: it downloads the project from GitHub and runs `scripts/train.py` — all training logic lives in `pneumonia/`.
Step-by-step instructions: [docs/KAGGLE_GUIDE.md](../docs/KAGGLE_GUIDE.md).

**Before running:** *Add Input* → dataset **Chest X-Ray Images (Pneumonia)** (paultimothymooney);
*Settings* → Accelerator **GPU T4 x2** (we use one GPU) and **Internet on**.

Edit only the settings cell below, then *Save Version → Save & Run All*. When it finishes, download
`results_<...>.zip` from the *Output* tab.

In [ ]:
# ---- Settings: the only cell you edit ----
MODELS = ["baseline_cnn"]   # baseline_cnn, vgg16, resnet50, densenet121, efficientnet_b0, mobilenetv3_large
SEEDS = [42]                # 42 for the first run of every model; then 43 and 44
DEBUG = False               # True = quick check: 16 images per class, 1 epoch
BRANCH = "main"             # git branch to train from
REPO_URL = "https://github.com/Denngerz/pneumonia_image_classification.git"

In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path

WORK = Path("/kaggle/working")
REPO = WORK / "pneumonia_image_classification"
RESULTS = WORK / "results"
RUNS = WORK / "runs"


def run(cmd: list[str], cwd: Path | None = None) -> None:
    """Run a command and stream its output into the notebook; stop on failure."""
    process = subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="")
    if process.wait() != 0:
        raise RuntimeError(f"Command failed ({process.returncode}): {' '.join(cmd)}")


run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"])

## 1. Get the code and the missing libraries

In [ ]:
if REPO.exists():
    shutil.rmtree(REPO)
run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO)])
run(["git", "-C", str(REPO), "log", "--oneline", "-1"])
# torch / torchvision are preinstalled with CUDA on Kaggle — never reinstall them.
run([sys.executable, "-m", "pip", "install", "-q", "timm>=1.0", "grad-cam>=1.5"])
run([sys.executable, "-c", "import torch, timm; print('torch', torch.__version__, '| timm', timm.__version__, "
     "'| cuda', torch.cuda.is_available())"])

## 2. Find the dataset

In [ ]:
# The attached dataset contains chest_xray/ twice (one nested copy); take the shortest path.
candidates = sorted(Path("/kaggle/input").glob("**/chest_xray/train/NORMAL"), key=lambda p: len(p.parts))
assert candidates, "Dataset not found: Add Input → 'Chest X-Ray Images (Pneumonia)'"
DATA_DIR = candidates[0].parent.parent
print(DATA_DIR)

## 3. Train, evaluate on the test set and make Grad-CAM pictures

In [ ]:
for model in MODELS:
    for seed in SEEDS:
        print(f"\n===== {model}, seed {seed} =====")
        cmd = [sys.executable, "scripts/train.py", "--config", f"configs/models/{model}.yaml", "--seed", str(seed),
               "--data-dir", str(DATA_DIR), "--results-dir", str(RESULTS), "--runs-dir", str(RUNS)]
        run(cmd + (["--debug"] if DEBUG else []), cwd=REPO)

## 4. Pack the results for download

In [ ]:
# Small result files (committed to git) → one zip in the Output tab.
results_dir = RUNS / "debug" if DEBUG else RESULTS
archive = shutil.make_archive(str(WORK / f"results_{'_'.join(MODELS)}_seed{'_'.join(map(str, SEEDS))}"
                                  f"{'_debug' if DEBUG else ''}"), "zip", results_dir)
print("Download from the Output tab:", archive)
for path in sorted(results_dir.rglob("metrics.json")):
    print(path.parent.name, path.read_text()[:400].replace("\n", " "))

# last.pt is only needed to resume a run; drop it to keep the notebook output small (best.pt stays).
for path in RUNS.rglob("last.pt"):
    path.unlink()
shutil.rmtree(REPO)